# Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Raw Results Dataframe

In [ ]:
def concatenate_raw_data(BASE_PATH, job_id_map=None):
    """Concatenate all `results.csv` files under BASE_PATH.

    Parameters
    - BASE_PATH: path to search for results.csv files
    - job_id_map: optional dict mapping full job_id strings to short abbreviations

    Returns concatenated DataFrame with an added `skill_char` and `job_abbrev` column.
    """
    dfs = []
    for path in Path(BASE_PATH).rglob("results.csv"):
        try:
            df = pd.read_csv(path)
            path_str = str(path)
            df["skill_char"] = path_str.split("skill_", 1)[1][0]

            if job_id_map:
                df["job"] = df["job_id"].map(job_id_map).fillna(df["job_id"].astype(str))
            else:
                df["job"] = df["job_id"].astype(str)

            dfs.append(df)
        except Exception as e:
            print(f"Failed to read {path}: {e}")
    raw_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    raw_df = raw_df.drop(columns=["temperature", "model_name"], errors="ignore")
    return raw_df


# Collapsed Dataframe

### Hilfsfunktionen — Statistik & Zusammenfassung


In [ ]:
def collapse_trials(raw_df):
    key_cols = [
        "job_id",
        "job",
        "skill_char",
        "variant_id",
        "candidate1_gender",
        "candidate2_gender",
        "candidate1_name",
        "candidate2_name",
        "male_position",
        "male_profile_type",
    ]
    collapsed = (
        raw_df.groupby(key_cols, as_index=False)
        .agg(
            **{
                "Winrate (f)": ("selected_gender", lambda s: (s == "W").mean()),
            }
        )
    )
    return collapsed

In [ ]:
def _finite_series(values):
    series = pd.Series(values, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return series

def _one_sample_summary(values, null_mean=0.5, confidence=0.95):
    series = _finite_series(values)
    n = int(series.shape[0])
    if n == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "cohens_d": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    mean = float(series.mean())
    std = float(series.std(ddof=1)) if n > 1 else np.nan

    if n > 1 and np.isfinite(std) and std > 0:
        t_stat, p_value = stats.ttest_1samp(series, popmean=null_mean)
        cohens_d = (mean - null_mean) / std
        sem = stats.sem(series, nan_policy="omit")
        if np.isfinite(sem) and sem > 0:
            ci_low, ci_high = stats.t.interval(confidence, df=n - 1, loc=mean, scale=sem)
        else:
            ci_low = ci_high = np.nan
    else:
        t_stat = p_value = cohens_d = np.nan
        ci_low = ci_high = np.nan

    return {
        "n": n,
        "mean": mean,
        "t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
        "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
        "cohens_d": float(cohens_d) if np.isfinite(cohens_d) else np.nan,
        "ci_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
        "ci_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
    }

def _significance_label(p_value, alpha=0.05):
    if pd.isna(p_value):
        return np.nan
    return "significant" if p_value < alpha else "not significant"

def _cohens_d_impact(cohens_d):
    if pd.isna(cohens_d):
        return np.nan
    magnitude = abs(cohens_d)
    if magnitude < 0.2:
        return "negligible"
    if magnitude < 0.5:
        return "small"
    if magnitude < 0.8:
        return "medium"
    return "large"

def _summarize_subset(df, label_col=None, label_value=None):
    subset = df if label_col is None else df[df[label_col] == label_value]
    raw = _one_sample_summary(subset["Winrate (f)"])

    return {
        "n_rows": int(len(subset)),
        "Winrate (f)": raw["mean"],
        "ci_low": raw["ci_low"],
        "ci_high": raw["ci_high"],
        "t_stat": raw["t_stat"],
        "p_value": raw["p_value"],
        "significance": _significance_label(raw["p_value"]),
        "cohens_d": raw["cohens_d"],
        "impact": _cohens_d_impact(raw["cohens_d"]),
    }

def summarize_winrates(df, group_col=None):
    if group_col is None:
        return pd.DataFrame([_summarize_subset(df)])

    rows = []
    for group_value, group_df in df.groupby(group_col, dropna=False):
        row = _summarize_subset(group_df)
        row[group_col] = group_value
        rows.append(row)

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(group_col).reset_index(drop=True)
    return result

### Export-Funktionen — PNG / LaTeX / CSV


In [ ]:
def df_to_png(df, path, fontsize=10, dpi=200):
    cell_text = df.fillna("").astype(str).values.tolist()
    cols = list(df.columns)
    nrows = len(cell_text)
    ncols = len(cols)
    fig_height = max(1, 0.35 * (nrows + 1))
    fig_width = max(2, 0.9 * ncols)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    table = ax.table(cellText=cell_text, colLabels=cols, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.2)
    plt.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)

def round_numeric(df, decimals=3):
    df2 = df.copy()
    num_cols = df2.select_dtypes(include=[float, int, 'number']).columns
    df2[num_cols] = df2[num_cols].round(decimals)
    return df2

def escape_underscores_for_latex(df):
    """Return a DataFrame where string columns have underscores escaped for LaTeX.

    This function first un-escapes any previously escaped underscores ("\\_")
    to avoid double-escaping, then escapes remaining underscores.
    Only object/string columns are modified; numeric columns are preserved.
    """
    df2 = df.copy()
    str_cols = df2.select_dtypes(include=['object', 'string']).columns

    def _esc(val):
        if pd.isna(val):
            return val
        s = str(val)
        s = s.replace(r'\_', '_')
        s = s.replace('_', r'\_')
        return s

    for c in str_cols:
        df2[c] = df2[c].apply(_esc)
    return df2

def export_results(overall_winrates, skill_group_winrates, job_abbrev_winrates, out_dir):

    EXPORT_DROP_COLS = ['n_rows', 'significance', 'impact', 'skill_char', 'job']

    overall_df_export = overall_winrates.copy()
    overall_df_export = round_numeric(overall_df_export, 3)
    overall_for_export = overall_df_export.drop(columns=EXPORT_DROP_COLS, errors='ignore')

    csv_path = out_dir / "overall_results.csv"
    tex_path = out_dir / "overall_results.tex"
    png_path = out_dir / "overall_results.png"
    overall_for_export.to_csv(csv_path, index=False, float_format="%.3f")
    overall_for_export.to_latex(tex_path, index=False, float_format="%.3f", escape=True)
    ndf = overall_for_export.copy()
    df_to_png(ndf, png_path, fontsize=10, dpi=200)

    def _standardize(df, group_type, key_col=None):
        df2 = df.copy()
        if group_type == 'skill':
            df2['group'] = 'skill ' + df2[key_col].astype(str)
        elif group_type == 'job':
            df2['group'] = df2[key_col].astype(str)
        else:
            df2['group'] = 'overall'
        return df2

    overall_std = _standardize(overall_winrates, 'overall', None)
    skill_std = _standardize(skill_group_winrates, 'skill', 'skill_char')
    job_std = _standardize(job_abbrev_winrates, 'job', 'job')
    combined = pd.concat([overall_std, skill_std, job_std], ignore_index=True, sort=False)
    cols_order = ['group'] + [c for c in combined.columns if c not in ('group')]
    combined = combined[cols_order]

    combined_export = round_numeric(combined, 3)
    combined_for_export = combined_export.drop(columns=EXPORT_DROP_COLS, errors='ignore')

    combined_csv = out_dir / "combined_results.csv"
    combined_tex = out_dir / 'combined_results.tex'
    combined_png = out_dir / 'combined_results.png'
    combined_for_export.to_csv(combined_csv, index=False, float_format="%.3f")
    combined_for_export.to_latex(combined_tex, index=False, float_format="%.3f", escape=True)
    ndf2 = combined_for_export.copy()
    df_to_png(ndf2, combined_png, fontsize=8, dpi=200)

    print('Export complete. Files written to', out_dir)


### Konfiguration & Batch-Verarbeitung der Modelle


In [ ]:
ROOT_FOLDERS = [
    #r"..\Results_Exp2_1",
    #r"..\Results_Exp2_2",
    #r"..\Results_Exp2_3",
    r"..\Results_Exp2_4",
]

MODELS = [
    "model_mistral_small_3_2_24b_instruct_2506",
    "model_openai_gpt_oss_120b",
]

MODEL_MAP = {
    "model_mistral_small_3_2_24b_instruct_2506" : "Mistral",
    "model_openai_gpt_oss_120b": "GPT OSS",
}

JOB_ID_MAP = {
    "entwickler_in": "Entw.",
    "HR": "HR",
    "product_owner": "Prod. Own.",
    "projektmanagement": "Proj. M",
    "scrum_master": "SCRUM",
    "security_architect": "Sec. Ar.",
    "senior_lead": "Sr. Lead",
    "team_lead": "Team L.",
}

for root in ROOT_FOLDERS:
    for model in MODELS:
        print(f"Processing model '{model}' in root '{root}'...")
        base_path = os.path.join(root, model)
        raw_df = concatenate_raw_data(base_path, job_id_map=JOB_ID_MAP)
        if raw_df.empty:
            print(f"No data found for model '{model}' in root '{root}'. Skipping.")
            continue
        collapsed_df = collapse_trials(raw_df)
        overall_winrates = summarize_winrates(collapsed_df)
        skill_group_winrates = summarize_winrates(collapsed_df, group_col="skill_char")
        job_abbrev_winrates = summarize_winrates(collapsed_df, group_col="job")

        out_dir = os.path.join(base_path, "Analysis")
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        export_results(overall_winrates, skill_group_winrates, job_abbrev_winrates, out_dir)

for root in ROOT_FOLDERS:
    root_path = Path(root)
    out_dir = root_path / "OverallResults"
    out_dir.mkdir(parents=True, exist_ok=True)

    dfs = []
    for model in MODELS:
        analysis_path = root_path / model / "Analysis"

        df = pd.read_csv(analysis_path / "overall_results.csv")
        df.insert(0, "model", MODEL_MAP.get(model, model))
        dfs.append(df)
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined.to_csv(out_dir / "overall_results_combined.csv", index=False, float_format="%.3f")
    df_combined.to_latex(out_dir / "overall_results_combined.tex", index=False, float_format="%.3f", escape=True)